# KEYSTONE H3 Runtime Bridge

Exposes the H3 ComfyUI server (`127.0.0.1:8188`) through a token-protected proxy and a real
Cloudflare tunnel, then prints the two values KEYSTONE needs.

- Unlike the previous gateway cell, this notebook never prints a placeholder URL: if the tunnel
  does not come up, it raises and points at its log.
- Every proxied request must carry `Authorization: Bearer <token>`.
- The token is access to your GPU session — keep it private and restart this cell to rotate it.
- Registration with a KEYSTONE control plane is deliberately absent: no control plane endpoint
  exists yet, so nothing is reported as registered.


In [ ]:
import json, urllib.request

with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=10) as response:
    stats = json.load(response)
print('ComfyUI reachable on 8188:', [d.get('name') for d in stats.get('devices', [])])


## 1. Install the bridge (written from the repo's versioned copy)

In [ ]:
from pathlib import Path

BRIDGE_SOURCE = r'''"""KEYSTONE H3 bridge — authenticated reverse proxy in front of Kaggle ComfyUI.

Runs inside the Kaggle runtime next to ComfyUI (127.0.0.1:8188) and listens on
127.0.0.1:8189. A cloudflared tunnel points at 8189, so every request from the
KEYSTONE API must present ``Authorization: Bearer <token>``.

    python keystone_h3_bridge.py --port 8189 --upstream http://127.0.0.1:8188

Token resolution: ``KEYSTONE_BRIDGE_TOKEN`` (Kaggle Secret / env), otherwise a
random token is generated, written to ``/tmp/keystone_bridge_token`` and printed.
Standard library only.
"""

from __future__ import annotations

import argparse
import http.server
import os
import secrets
import socketserver
import sys
import tempfile
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

TOKEN_FILE = Path(tempfile.gettempdir()) / "keystone_bridge_token"
HOP_HEADERS = {"host", "connection", "keep-alive", "proxy-authenticate", "te", "trailers",
               "transfer-encoding", "upgrade", "content-length", "authorization"}


def resolve_token() -> str:
    token = os.getenv("KEYSTONE_BRIDGE_TOKEN", "").strip()
    if token:
        TOKEN_FILE.write_text(token)
        return token
    if TOKEN_FILE.exists():
        existing = TOKEN_FILE.read_text().strip()
        if existing:
            return existing
    token = secrets.token_urlsafe(32)
    TOKEN_FILE.write_text(token)
    return token


def make_handler(upstream: str, token: str):
    base = upstream.rstrip("/")

    class Handler(http.server.BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"

        def log_message(self, fmt: str, *args) -> None:  # keep Kaggle logs readable
            sys.stderr.write("bridge %s - %s\n" % (self.address_string(), fmt % args))

        def _authorized(self) -> bool:
            header = self.headers.get("Authorization", "")
            return header.startswith("Bearer ") and secrets.compare_digest(header[7:].strip(), token)

        def _json(self, status: int, payload: bytes, content_type: str = "application/json") -> None:
            self.send_response(status)
            self.send_header("Content-Type", content_type)
            self.send_header("Content-Length", str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)

        def _proxy(self) -> None:
            if self.path.split("?")[0] == "/keystone/health":
                body = ('{"ok":true,"upstream":"%s"}' % base).encode()
                return self._json(200, body)

            if not self._authorized():
                return self._json(401, b'{"error":"unauthorized"}')

            length = int(self.headers.get("Content-Length") or 0)
            payload = self.rfile.read(length) if length else None
            headers = {key: value for key, value in self.headers.items() if key.lower() not in HOP_HEADERS}
            headers["Host"] = urllib.parse.urlparse(base).netloc

            request = urllib.request.Request(
                base + self.path, data=payload, headers=headers, method=self.command
            )
            try:
                response = urllib.request.urlopen(request, timeout=1800)
            except urllib.error.HTTPError as exc:
                data = exc.read()
                return self._json(exc.code, data, exc.headers.get("Content-Type", "application/json"))
            except urllib.error.URLError as exc:
                return self._json(502, ('{"error":"upstream unreachable: %s"}' % exc).encode())

            with response:
                self.send_response(response.status)
                for key, value in response.headers.items():
                    if key.lower() in {"transfer-encoding", "connection", "content-length"}:
                        continue
                    self.send_header(key, value)
                data = response.read()
                self.send_header("Content-Length", str(len(data)))
                self.end_headers()
                self.wfile.write(data)

        do_GET = _proxy
        do_POST = _proxy
        do_HEAD = _proxy

    return Handler


class ThreadingServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True
    allow_reuse_address = True


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8189)
    parser.add_argument("--upstream", default="http://127.0.0.1:8188")
    args = parser.parse_args()

    token = resolve_token()
    server = ThreadingServer((args.host, args.port), make_handler(args.upstream, token))
    print(f"KEYSTONE H3 bridge on http://{args.host}:{args.port} -> {args.upstream}", flush=True)
    print(f"KEYSTONE_BRIDGE_TOKEN={token}", flush=True)
    server.serve_forever()


if __name__ == "__main__":
    main()
'''
Path('/tmp/keystone_h3_bridge.py').write_text(BRIDGE_SOURCE, encoding='utf-8')
print('bridge source bytes:', len(BRIDGE_SOURCE))


## 2. Start the bridge and the tunnel, then prove it through the public URL

In [ ]:
import json, os, re, secrets, shutil, subprocess, time, urllib.request
from pathlib import Path

TOKEN_FILE = Path('/tmp/keystone_bridge_token')
token = os.environ.get('KEYSTONE_BRIDGE_TOKEN') or (
    TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else secrets.token_urlsafe(32)
)
TOKEN_FILE.write_text(token)

bridge_log = open('/tmp/keystone_bridge.log', 'w')
subprocess.Popen(
    ['python', '/tmp/keystone_h3_bridge.py', '--port', '8189', '--upstream', 'http://127.0.0.1:8188'],
    stdout=bridge_log, stderr=subprocess.STDOUT,
    env=dict(os.environ, KEYSTONE_BRIDGE_TOKEN=token),
)
for _ in range(20):
    time.sleep(0.5)
    try:
        urllib.request.urlopen('http://127.0.0.1:8189/keystone/health', timeout=2).read()
        break
    except Exception:
        continue
else:
    raise RuntimeError(Path('/tmp/keystone_bridge.log').read_text())
print('bridge ready on 8189 -> 8188')

if not shutil.which('cloudflared'):
    subprocess.run(
        'curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
        ' -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
        shell=True, check=True,
    )

tunnel_log = open('/tmp/cloudflared_8189.log', 'w')
subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8189', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
url = None
for _ in range(40):
    time.sleep(1)
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',
                      Path('/tmp/cloudflared_8189.log').read_text(errors='ignore'))
    if match:
        url = match.group(0)
        break
if not url:
    raise RuntimeError('No tunnel URL. Inspect /tmp/cloudflared_8189.log')
print('tunnel:', url)

request = urllib.request.Request(url + '/keystone/health', headers={'Authorization': f'Bearer {token}'})
with urllib.request.urlopen(request, timeout=30) as response:
    print('public health:', response.status, response.read()[:120])
request = urllib.request.Request(url + '/system_stats', headers={'Authorization': f'Bearer {token}'})
with urllib.request.urlopen(request, timeout=30) as response:
    print('public ComfyUI:', response.status)

print('
Paste into KEYSTONE .env:
')
print(f'KEYSTONE_COMFY_URL={url}')
print(f'KEYSTONE_COMFY_API_KEY={token}')
print('KEYSTONE_COMFY_WORKFLOW=workflows/h3-ref2va.json')


## Next step

Copy the two printed lines into the KEYSTONE repo `.env`:

```
KEYSTONE_COMFY_URL=https://<tunnel>.trycloudflare.com
KEYSTONE_COMFY_API_KEY=<token>
KEYSTONE_COMFY_WORKFLOW=workflows/h3-ref2va.json
```

Restart the API so the values load, then send an Animate request with 1–2 reference images.
The tunnel dies with the Kaggle session; rerun this cell and update `.env`.
